In [1]:
%matplotlib inline
import pandas as pd
pd.options.mode.chained_assignment = None
import yaml
from scripts.prep_TFBS import prep_tfbs, roi_kmers
from scripts.process_TFBS_damages import proc_TFBS_dams, count_damages
from scripts.damage_formation.analyze_TFBS_damage import predict_TFBS_damage, analyze_TFBS_damage

In [2]:
with open("config/pipeline_config.yaml", "r") as f:
    config = yaml.safe_load(f)

genome_dir = config["folders"]["genome_dir"]
main_dir = config["folders"]["main_dir"]
cpd_seq_data_dir = config["folders"]["cpd_seq_data_dir"]

tf_window_size = config["parameters"]["tf_window_size"]
kmer = config["parameters"]["kmer_length"]
damage_formation_analysis_window = config["parameters"]["damage_formation_analysis_window"]
tfbs_buffer_size = config["parameters"]["tfbs_buffer_size"]

archetype_file = config["files"]["archetype_file"]
dhs_file = config["files"]["dhs_file"]
motif_cluster_file = config["files"]["motif_cluster_file"]

In [3]:
exps = ['WT_CSB_6J_0hr_CPD_1bp_sorted','WT_CSB_6J_6hr_CPD_1bp_sorted', 'XPC_12J_NakedDNA_S22_CPD_1bp_sorted']

In [4]:
clusters = pd.read_csv(f'{main_dir}/data/TF_cluster_motifs.csv')
cluster_records = clusters.to_records(index=False)

In [5]:
for tf_cluster, tf_len, tf in cluster_records:
    #Prepare TFBS windows
    #prep_tfbs(tf_cluster, tf_len, tf, f"{main_dir}/results/TFBS", genome_dir, archetype_file, tf_window_size, f"{main_dir}/data")

    #Count NYYN by position across aggregated TFBS windows
    #roi_kmers(f"{tf}_top_{tf_window_size}", kmer, f"{main_dir}/results/TFBS")

    for exp in exps:
        #Intersect damages in TFBS windows
        #proc_TFBS_dams(f'{tf}_top_{tf_window_size}_seq', kmer, exp, cpd_seq_data_dir, f"{main_dir}/results/TF_damage_data", f"{main_dir}/results/TFBS", genome_dir)

        #Count aggregate damage by position in TFBS windows
        #count_damages(f'{tf}_top_{tf_window_size}_seq', kmer, exp, f"{main_dir}/results/TF_damage_data", f"{main_dir}/results/TFBS")

        #Calculate predicted damage by position in TFBS windows
        #predict_TFBS_damage(f'{tf}_top_{tf_window_size}', exp, kmer, tf_window_size,dhs_file, f"{main_dir}/results/background",f"{main_dir}/results/TF_damage_data", f"{main_dir}/results/TFBS")

        #Analyze damage formation in TFBS windows
        analyze_TFBS_damage(tf, f'{tf}_top_{tf_window_size}', exp, kmer, tf_window_size, (tf_len//2)+damage_formation_analysis_window, (tf_len//2)+tfbs_buffer_size, dhs_file, f"{main_dir}/results/TF_damage_data", f"{main_dir}/results/analysis/damage_formation")

In [6]:
#Concatenate results
for exp in exps:
    !cat results/analysis/damage_formation/ETS_1_top_100_{exp}_{dhs_file}_6mer_pvals_corrected.csv results/analysis/damage_formation/SOX_1_top_100_{exps[0]}_{dhs_file}_6mer_pvals_corrected.csv > results/analysis/damage_formation/all_tfs_top_100_{exp}_{dhs_file}_6mer_pvals_corrected.csv